# Chapter 1 — ARM Architecture Overview for x86 Engineers

## Concept
ARM64 (AArch64) ISA differences vs x86-64: load/store architecture, fixed-width
32-bit instructions, 31 general-purpose integer registers (X0–X30), link register
(X30/LR), stack pointer (SP), PC not a general register, condition flags (NZCV),
calling convention (AAPCS64), and Exception Levels (EL0–EL3).

## Lab Objectives
1. Boot an ARM64 VM on QEMU `virt` machine.
2. Query vCPU state via QMP (`query-cpus-fast`, `query-version`).
3. Confirm EL2 entry — QEMU boots to EL2 by default before handing off to EL1.
4. Assert CPU model is cortex-a76 (or aarch64-generic if max selected).

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=[],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Query QEMU version ───────────────────────────────────────────────
ver = qmp.query_version()
print("QEMU version dict:", json.dumps(ver, indent=2))


In [ ]:
# ── Step 2: Query vCPUs (query-cpus-fast) ────────────────────────────────────
import json
cpus = qmp.query_cpus()
print(f"vCPU count : {len(cpus)}")
for c in cpus:
    print(f"  CPU {c['cpu-index']}: arch={c.get('arch','n/a')}  "
          f"thread_id={c.get('thread-id','n/a')}")


In [ ]:
# ── Step 3: Query machine info via HMP bridge ─────────────────────────────────
machine_info = qmp.human_monitor_command("info cpus")
print("HMP info cpus:\n", machine_info)


In [ ]:
# ── Step 4: Read serial boot log — confirm EL2 entry ─────────────────────────
# The QEMU virt machine starts firmware (UEFI/ATF stub) in EL2.
# The serial log will contain UEFI output before the kernel banner.
boot_log_snippet = sc.run_command(
    "dmesg | head -40", timeout=15
)
print(boot_log_snippet[:1000])


In [ ]:
# ── Step 5: Confirm kernel is running at EL1 ─────────────────────────────────
el_info = sc.run_command(
    "cat /proc/cpuinfo | grep -i 'CPU implementer\|CPU architecture\|CPU part'",
    timeout=10
)
print(el_info)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_true(launcher.is_running(), "QEMU process is alive",
            detail="Subprocess poll() is None")

assert_true(len(cpus) >= 1, "query-cpus-fast returns ≥ 1 vCPU",
            detail=str(cpus),
            action="Check -smp parameter in QEMULauncher")

assert_equal(len(cpus), SMP, f"vCPU count == SMP ({SMP})",
             action="SMP mismatch — check launcher config")

assert_contains(str(cpus), r"cpu-index",
                "query-cpus-fast response has cpu-index field",
                action="Unexpected QMP response structure")

assert_contains(str(ver), r"qemu",
                "QEMU version string present",
                action="QMP version query returned unexpected data")

# CPU model check (cortex-a76 or aarch64-generic)
cpu_model_raw = qmp.human_monitor_command("info registers")
# Just verify the HMP command returns something non-empty
assert_true(len(cpu_model_raw) > 10,
            "HMP 'info registers' returns register state",
            detail=cpu_model_raw[:80])

assert_contains(el_info, r"CPU implementer",
                "Guest /proc/cpuinfo shows ARM CPU implementer",
                action="Guest may not be ARM64 — check -cpu flag")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| QEMU process alive | Yes | Subprocess spawned without error |
| query-cpus-fast ≥ 1 vCPU | 1 vCPU | Matches -smp 1 |
| cpu-index field present | Yes | QMP schema validation |
| QEMU version string | Contains 'qemu' | Version negotiation |
| HMP info registers | Non-empty | EL state readable |
| /proc/cpuinfo ARM implementer | Present | Kernel sees ARM64 CPU |

## Next Steps
→ **Chapter 2**: Memory Model — inspect guest physical memory map and hot-plug RAM.
